In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
%run /Workspace/consolidated-pipeline/1st_setup/Utilities

In [0]:
dbutils.widgets.text("Catalog","fmcg","catalog")
dbutils.widgets.text("Data_Source","gross_price","data_source")
catalog=dbutils.widgets.get("Catalog")
data_source=dbutils.widgets.get("Data_Source")
print(catalog,data_source)

In [0]:
base_path= f"s3://amzn-s3-sportsbar/{data_source}/*.csv"
print(base_path)

# Bronze Processing

In [0]:
df= (
  spark.read.format("csv")
  .option("header",True)
  .option("inferSchema",True)
  .option("mode","PERMISSIVE")
  .option("multiline",True)
  .load(base_path)
  .withColumn("read_timestamp",F.current_timestamp())
  .select("*","_metadata.file_name","_metadata.file_size")
  )

In [0]:
df.printSchema()

In [0]:
display(df.limit(10))

In [0]:
df.write.format("delta")\
    .option("delta.enableChangeDataFeed",True)\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

# Silver Processing

In [0]:
df_bronze= spark.sql(f"Select * from {catalog}.{bronze_schema}.{data_source}")
df.show(5)

Tranformations : normalize month field

In [0]:
df_bronze.select("month").distinct().show()

In [0]:
# same same
# 1️. Parse `month` from multiple possible formats
date_formats = ["yyyy/MM/dd", "dd/MM/yyyy", "yyyy-MM-dd", "dd-MM-yyyy"]

df_silver = df_bronze.withColumn(
    "month",
    F.coalesce(
        F.try_to_date(F.col("month"), "yyyy/MM/dd"),
        F.try_to_date(F.col("month"), "dd/MM/yyyy"),
        F.try_to_date(F.col("month"), "yyyy-MM-dd"),
        F.try_to_date(F.col("month"), "dd-MM-yyyy")
    )
)

In [0]:
df_silver.select("month").distinct().show()

In [0]:
df_silver.filter(F.col("month").isNull()).show()

In [0]:
# Handling the gross price
df_silver = df_silver.withColumn(
    "gross_price",
    F.when(
        F.col("gross_price").rlike("^-?\d+(\.\d+)?$"),
        F.when(
            F.col("gross_price").cast("double") < 0 , -1 * F.col("gross_price").cast("double")
        ).otherwise(F.col("gross_price").cast("double"))  
           ).otherwise(0)
)

In [0]:
df_silver.select("gross_price").show()

In [0]:
df_products = spark.table("fmcg.silver.products")
df_joined = df_silver.join(df_products.select("product_code","product_id"),
    on ="product_id",
    how ="inner"
)
df_joined.show(5)

In [0]:
df_joined.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed",True)\
    .option("mergeSchema",True)\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

# Gold Processing

In [0]:
df_silver_table = spark.sql(f"Select * From {catalog}.{silver_schema}.{data_source}")
df_gold = df_silver_table.select("product_code","month","gross_price")


In [0]:
display(df_gold.limit(5))

In [0]:
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
# with same product_code we have many gross_price. We will only keep 1 price per product , that too latest price.
df_gold = (
    df_gold.withColumn("year", F.year("month"))
    .withColumn("isZero", F.when(F.col("gross_price")==0,1).otherwise(0))
)
w = (
    Window.partitionBy("product_code","year")
    .orderBy(F.col("isZero"),F.col("month").desc())
)
df_gold_latest_price = df_gold.withColumn("rnk",F.row_number().over(w))\
    .filter(F.col("rnk")==1)

In [0]:
display(df_gold_latest_price)

In [0]:
df_gold_latest_price = df_gold_latest_price.select("product_code","gross_price","year")\
    .withColumnRenamed("gross_price","price_inr")
df_gold_latest_price = df_gold_latest_price.withColumn("year",F.col("year").cast("string"))
df_gold_latest_price.show(5)

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_gross_price")
delta_table.alias("target")\
    .merge(
        source = df_gold_latest_price.alias("source"),
        condition = "target.product_code = source.product_code"
    ).whenMatchedUpdate(
        set = {
            'price_inr' : 'source.price_inr',
            'year' : 'source.year'
        }
    ).whenNotMatchedInsert(
        values = {
            'product_code' : 'source.product_code',
            'price_inr' : 'source.price_inr',
            'year' : 'source.year'
        }
    ).execute()